# Day 3 — Planner and fan-out

A planner turns a clarified request into tasks and dependencies.

First, build and validate a deterministic plan without an LLM.
Later, compare it with an LLM-generated plan using the same validators.

In [1]:
from aegis.nodes.planner import plan_deterministically
from aegis.schemas import ResearchRequest, ResearchScope

request = ResearchRequest(
    objective=(
        "Compare Player A and Player B on scoring and creativity "
        "during the 2025 Demo League."
    ),
    subjects=["Player A", "Player B"],
    factors=["scoring", "creativity"],
    mode="compare",
    scope=ResearchScope(
        time_period="2025",
        coverage="Demo League",
    ),
)

plan = plan_deterministically(request)

for step in plan.steps:
    print(
        f"{step.step_id:<18} "
        f"worker={step.worker:<8} "
        f"subject={step.subject:<10} "
        f"factor={step.factor:<10} "
        f"depends_on={step.depends_on}"
    )

evidence:s1:f1     worker=evidence subject=Player A   factor=scoring    depends_on=[]
analysis:s1:f1     worker=analysis subject=Player A   factor=scoring    depends_on=['evidence:s1:f1']
evidence:s1:f2     worker=evidence subject=Player A   factor=creativity depends_on=[]
analysis:s1:f2     worker=analysis subject=Player A   factor=creativity depends_on=['evidence:s1:f2']
evidence:s2:f1     worker=evidence subject=Player B   factor=scoring    depends_on=[]
analysis:s2:f1     worker=analysis subject=Player B   factor=scoring    depends_on=['evidence:s2:f1']
evidence:s2:f2     worker=evidence subject=Player B   factor=creativity depends_on=[]
analysis:s2:f2     worker=analysis subject=Player B   factor=creativity depends_on=['evidence:s2:f2']


In [2]:
# Two subjects × two factors = four independent retrieval tasks.
retrieval_steps = [
    step for step in plan.steps
    if step.worker == "evidence"
]

# Each retrieval task has one dependent analysis task.
analysis_steps = [
    step for step in plan.steps
    if step.worker == "analysis"
]

assert len(retrieval_steps) == 4
assert len(analysis_steps) == 4
assert len(plan.steps) == 8

print("PASS: 4 retrieval tasks and 4 dependent analysis tasks")

PASS: 4 retrieval tasks and 4 dependent analysis tasks


In [3]:
from graphlib import TopologicalSorter

# The sorter receives each task and its prerequisites.
sorter = TopologicalSorter(
    {
        step.step_id: set(step.depends_on)
        for step in plan.steps
    }
)
sorter.prepare()

# Initially, only retrieval tasks should be ready.
first_wave = tuple(sorted(sorter.get_ready()))

assert set(first_wave) == {
    step.step_id for step in retrieval_steps
}

print("First wave:", first_wave)

# Simulate successful completion. No workers actually run here.
sorter.done(*first_wave)

second_wave = tuple(sorted(sorter.get_ready()))

assert set(second_wave) == {
    step.step_id for step in analysis_steps
}

print("Second wave:", second_wave)
print("PASS: analysis becomes ready after retrieval")

First wave: ('evidence:s1:f1', 'evidence:s1:f2', 'evidence:s2:f1', 'evidence:s2:f2')
Second wave: ('analysis:s1:f1', 'analysis:s1:f2', 'analysis:s2:f1', 'analysis:s2:f2')
PASS: analysis becomes ready after retrieval


## Compare deterministic and LLM planning

Both planners receive the same six valid requests.

Compare task meaning and dependency structure, not generated task IDs
or wording. A valid plan does not prove that the required evidence exists.

In [4]:
import json
import os
from decimal import Decimal
from pathlib import Path

from dotenv import load_dotenv

# Find the repository root from either the root or notebooks directory.
project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").is_file()
        and (path / "src" / "aegis").is_dir()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("Could not locate the Aegis project root")

os.chdir(project_root)
load_dotenv(project_root / ".env", override=True)

from aegis.config import get_settings
from aegis.llm.callbacks import new_run_budget
from aegis.nodes.planner import (
    plan_deterministically,
    plan_with_llm,
)
from aegis.schemas import ResearchRequest

get_settings.cache_clear()
settings = get_settings()

In [5]:
dataset_path = project_root / "evals" / "datasets" / "v1.jsonl"

cases = [
    json.loads(line)
    for line in dataset_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

valid_cases = [
    case for case in cases
    if case["category"] == "valid"
]

assert len(valid_cases) == 6

for case in valid_cases:
    print(case["id"])

football_research
football_compare
football_rank
platform_research
platform_compare
platform_rank


In [6]:
def plan_signature(plan):
    """Compare tasks and dependencies without relying on generated IDs."""

    def task_key(step):
        return (
            step.worker,
            step.subject.casefold(),
            step.factor.casefold(),
        )

    steps_by_id = {
        step.step_id: step
        for step in plan.steps
    }

    # Keep a list, rather than a set, so duplicate tasks remain visible.
    return sorted(
        (
            task_key(step),
            tuple(
                sorted(
                    task_key(steps_by_id[dependency])
                    for dependency in step.depends_on
                )
            ),
        )
        for step in plan.steps
    )

In [7]:
# Use one shared tracker, not a fresh budget for each case.
evaluation_settings = settings.model_copy(
    update={
        "budget_limit_usd": min(
            settings.budget_limit_usd,
            Decimal("0.10"),
        )
    }
)

tracker, base_config = new_run_budget(evaluation_settings)

# Keep successful results in memory even if a later case fails.
planner_results = []
llm_plans = {}
deterministic_plans = {}

try:
    for case in valid_cases:
        request = ResearchRequest.model_validate(
            case["input"]["request"]
        )

        deterministic_plan = plan_deterministically(request)

        config = {
            **base_config,
            "run_name": f"planner-{case['id']}",
            "tags": ["day-3", "planner-comparison"],
            "metadata": {
                "case_id": case["id"],
                "experiment": "planner-structure",
            },
        }

        cost_before = tracker.spent_usd

        llm_plan = plan_with_llm(
            request,
            config=config,
            retrieval_worker="evidence",
        )

        equivalent = (
            plan_signature(deterministic_plan)
            == plan_signature(llm_plan)
        )

        deterministic_plans[case["id"]] = deterministic_plan
        llm_plans[case["id"]] = llm_plan

        result = {
            "case_id": case["id"],
            "deterministic_steps": len(deterministic_plan.steps),
            "llm_steps": len(llm_plan.steps),
            "equivalent": equivalent,
            "cost_usd": str(tracker.spent_usd - cost_before),
        }
        planner_results.append(result)

        print(result)

        if not equivalent:
            raise AssertionError(
                f"Plan structure differs for {case['id']}"
            )
finally:
    # Show recorded spending even if parsing or validation fails.
    print("Recorded model calls:", len(tracker.calls))
    print("Recorded total USD:", tracker.spent_usd)

assert len(planner_results) == 6
assert all(row["equivalent"] for row in planner_results)

print("PASS: both planners agree structurally on all 6 valid cases")

{'case_id': 'football_research', 'deterministic_steps': 2, 'llm_steps': 2, 'equivalent': True, 'cost_usd': '0.0032875'}
{'case_id': 'football_compare', 'deterministic_steps': 8, 'llm_steps': 8, 'equivalent': True, 'cost_usd': '0.0061575'}
{'case_id': 'football_rank', 'deterministic_steps': 6, 'llm_steps': 6, 'equivalent': True, 'cost_usd': '0.0048975'}
{'case_id': 'platform_research', 'deterministic_steps': 2, 'llm_steps': 2, 'equivalent': True, 'cost_usd': '0.0030950'}
{'case_id': 'platform_compare', 'deterministic_steps': 8, 'llm_steps': 8, 'equivalent': True, 'cost_usd': '0.0056800'}
{'case_id': 'platform_rank', 'deterministic_steps': 8, 'llm_steps': 8, 'equivalent': True, 'cost_usd': '0.0053975'}
Recorded model calls: 6
Recorded total USD: 0.0285150
PASS: both planners agree structurally on all 6 valid cases


## When is the LLM planner useful?

For the current fixed rule—one retrieval and one analysis task per
subject/factor—the deterministic planner is sufficient and avoids an
extra model call.

An LLM planner may become useful when requests require semantic task
decomposition or choosing among sources with different capabilities.

Before enabling source selection, the planner needs a description of
available sources and workers. Its choices must still be validated.

The deterministic planner is available as a fallback when its fixed
routing assumptions apply. This experiment does not automatically
substitute it for a failed LLM plan.

## Routing graph with parallel stub workers

Validate → clarify → plan → dispatch evidence tasks → collect.

This experiment uses no LLM calls and produces only synthetic stub
evidence. Analysis tasks remain pending.

In [8]:
from aegis.graphs.routing import build_routing_graph
from aegis.state import RunStatus, bundle_from_state

graph = build_routing_graph()

raw_request = {
    "objective": (
        "Compare three fictional players on scoring and creativity."
    ),
    "subjects": ["Player A", "Player B", "Player C"],
    "factors": ["scoring", "creativity"],
    "mode": "compare",
    "scope": {
        "time_period": "2025",
        "coverage": "Demo League",
    },
}

# Pass only raw input. The graph constructs its validated state.
result = graph.invoke({"raw_request": raw_request})

bundle = bundle_from_state(result)

worker_events = [
    event
    for event in result["node_events"]
    if event["node"] == "stub_worker"
]

collector_events = [
    event
    for event in result["node_events"]
    if event["node"] == "collect"
]

assert len(worker_events) == 6
assert len(bundle.evidence) == 6
assert len(bundle.sources) == 1
assert len(collector_events) == 1
assert result["status"] == RunStatus.DEGRADED

print("Worker executions:", len(worker_events))
print("Evidence records:", len(bundle.evidence))
print("Source records:", len(bundle.sources))
print("Collector executions:", len(collector_events))
print("Status:", result["status"].value)
print("PASS: six worker results merged and collected")

Worker executions: 6
Evidence records: 6
Source records: 1
Collector executions: 1
Status: degraded
PASS: six worker results merged and collected


## Routing experiment result

Three subjects × two factors produced six worker executions.

The reducers preserved six distinct evidence records while deduplicating
the shared synthetic source. The collector executed once.

Invalid requests stop before clarification. Incomplete requests stop
before planning. Analysis tasks remain unexecuted in this prototype.